# Create MERlin data organization

**Variable z per FOV:** notebook 04 wrote one `frame-table-bits-{tier}-*.csv`
per tier. The data organization declares the round's full z-range (MERlin
reads each FOV's real depth from its raw file's frame count), so the bits
frame table used is the one with the most frames, the deepest tier. With one
bits frame table, that is the one used.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

MERCI_DIR    = Path(os.getcwd()).parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/before_imaging/multi_z/)
SAMPLE_DIR   = MERCI_DIR.parent                  # experiment root, e.g. LT027_saving_time/
METADATA_DIR = SAMPLE_DIR / "metadata"
SETTINGS_DIR = SAMPLE_DIR / "settings"
MERLIN_DIR   = SAMPLE_DIR / "merlin"
DATA_ORG_DIR = MERLIN_DIR / "dataorganization"
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.data_organization import create_data_organization
from MERci.acquisition.dave import annotate_dave_with_round_info, dave_config_filename
from MERci.common.experiment_info import resolve_sample_identity

MICROSCOPE  = "ST2"
# SAMPLE_NAME names the output files, as in the other notebooks.
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)

print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")
print(f"METADATA_DIR : {METADATA_DIR}")
print(f"SETTINGS_DIR : {SETTINGS_DIR}")
print(f"DATA_ORG_DIR : {DATA_ORG_DIR}")

## Auto-detect frame tables

Cells: `frame-table-cells-*`. Bits: the `frame-table-bits-*` with the most
rows (the deepest tier). Transit tables are ignored. Set `BITS_FT`/`CELLS_FT`
to override.

In [ ]:
ft_files = sorted(METADATA_DIR.glob("frame-table-*.csv"))

bits_ft_candidates = [f for f in ft_files if f.name.startswith("frame-table-bits-")]
cells_ft_candidates = [f for f in ft_files if f.name.startswith("frame-table-cells-")]

print("Frame tables found:")
for f in ft_files:
    ft = pd.read_csv(f, index_col=0)
    colors = sorted(ft["color"].dropna().unique().astype(int))
    print(f"  {f.name}  ({len(ft)} frames)  →  colors: {colors}")

if len(bits_ft_candidates) > 1:
    print(f"\n{len(bits_ft_candidates)} bits frame tables found -- variable-z-per-FOV round "
          f"(notebook 04); picking the one with the most frames (deepest tier) for the "
          f"data organization's declared z-range.")

BITS_FT  = max(bits_ft_candidates,  key=lambda f: len(pd.read_csv(f, index_col=0)))
CELLS_FT = next(f for f in cells_ft_candidates)

print(f"\nBits  frame table (deepest, if multiple) : {BITS_FT.name}")
print(f"Cells frame table                        : {CELLS_FT.name}")

## Round – bit – color mapping

Read back from notebook 05's `round_bit_color_map.csv`. Re-run 05 to change
it.

In [ ]:
# Saved by notebook 05.
rbc_path = METADATA_DIR / "round_bit_color_map.csv"
if not rbc_path.exists():
    raise FileNotFoundError(
        f"{rbc_path} not found — run notebook 05 first "
        f"(it now defines the round–bit–color mapping and N_HYBS)."
    )
rbc_df          = pd.read_csv(rbc_path)
round_bit_color = [(int(r), int(b), int(c))
                   for r, b, c in rbc_df[["round", "bit", "color"]].itertuples(index=False, name=None)]
print(f"Loaded {len(round_bit_color)} (round, bit, color) rows from {rbc_path.name}")
print(rbc_df.to_string(index=False))

## Series patterns from round_info.csv

The bits and cells series patterns, for the data organization's `imageType`
and `imageRegExp`. Re-run notebook 05 first if `round_info.csv` is out of
date.

In [ ]:
round_info_path = METADATA_DIR / "round_info.csv"
round_info      = pd.read_csv(round_info_path)

# Pick the bits/cells series by imaging_type (skips transit); older round_info
# files without it are matched by name.
MULTI_BOUNDARY = "segment" in round_info.columns
if "imaging_type" in round_info.columns:
    itype        = round_info["imaging_type"].astype(str).str.lower()
    bits_rows    = round_info[itype == "bits"]
    cells_rows   = round_info[itype == "cells"]
else:
    bits_rows    = round_info[~round_info["series"].str.contains("cells")]
    cells_rows   = round_info[ round_info["series"].str.contains("cells")]

bits_series  = bits_rows.iloc[0]["series"]
cells_series = cells_rows.iloc[0]["series"]

print(f"Multi-boundary layout : {MULTI_BOUNDARY}")
print(f"Bits  series (sample) : {bits_series}")
print(f"Cells series (sample) : {cells_series}")

if "tissue_thickness" in round_info.columns and (round_info["tissue_thickness"] == "multi").any():
    print("\nNOTE: this round is variable-z-per-FOV (tissue_thickness=\"multi\"). MERlin's "
          "ragged-z-stack decoding needs --allow-ragged-z-stacks (see notebook 09) and, as of "
          "this writing, is still incomplete upstream -- confirm with whoever maintains your "
          "MERlin checkout before relying on this for a real decode.")

if MULTI_BOUNDARY:
    tissues = sorted(round_info["tissue"].dropna().astype(int).unique())
    print(f"\nTissues present       : {tissues}")
    print("NOTE: multi-tissue MERlin analysis is per tissue, and each boundary is a\n"
          "distinct movie (imageType). The cell below builds ONE data-organization from\n"
          "the representative series above; for a per-tissue / per-boundary MERlin run,\n"
          "confirm the intended workflow before relying on this file.")

## Build and save data organization

In [ ]:
readouts  = pd.read_csv(MERCI_DIR / "data" / "readouts.csv")
ft_bits   = pd.read_csv(BITS_FT,  index_col=0)
ft_cells  = pd.read_csv(CELLS_FT, index_col=0)

data_org = create_data_organization(
    bits_frame_table  = ft_bits,
    cells_frame_table = ft_cells,
    round_bit_color   = round_bit_color,
    readouts          = readouts,
    bits_series       = bits_series,
    cells_series      = cells_series,
    include_dapi      = True,
)

out_name = f"data_organization_{MICROSCOPE.upper()}_{SAMPLE_NAME}.csv"
out_path = METADATA_DIR / out_name
data_org.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"\n{len(data_org)} rows  ({len(data_org)-1} bits + DAPI)")
data_org

## Annotate Dave XML with bit information

Adds an XML comment naming each round's bits to notebook 06's Dave config, on
the fluidics loop before each bits round (imaging rounds 2..N+1; round 1 is
cells). Re-run when the mapping changes.

In [ ]:
N_HYBS    = int(rbc_df["round"].max())
dave_path = SETTINGS_DIR / dave_config_filename(MICROSCOPE, N_HYBS, SAMPLE_NAME)

if not dave_path.exists():
    print(f"No {dave_path.name} found in settings/ — run notebook 06 first.")
else:
    print(f"Annotating: {dave_path.name}")
    # round_bit_color rounds are 1..N; the Dave recipe's bits are imaging rounds 2..N+1.
    annotate_rbc = [(r + 1, bit, color) for (r, bit, color) in round_bit_color]
    annotate_dave_with_round_info(dave_path, annotate_rbc)
    print("Done. Preview of annotated file:")
    with open(dave_path, encoding="ISO-8859-1") as fh:
        print(fh.read())